# 🚀**Aircheck Workshop: Using Machine Learning to Find Hits**🧬
Welcome to the Aircheck ML Model Training and Prediction Notebook! This notebook walks you through the various steps involved in training, evaluating, and screening small molecules using machine learning models built on chemical fingerprints.

![Alt Text](https://drive.google.com/uc?export=view&id=1E_Ivax_l5bvk2A8HKjBw5NYDgNMZ7wRa)


--------------------------------------------------------------------------------

## **🟣 1: Install and Import Dependencies**
In this section, we install the necessary packages for chemical data processing and machine learning.

In [ ]:
# Install the requirements:
!pip install pandas numpy lightgbm rdkit



In [ ]:

# Import libraries:
import pandas as pd
import numpy as np
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import AllChem
import os

---

## **🟣 2: Load and Prepare Data**

Next, we will load ....

In [ ]:
# Mount Google Drive in Colab to access and save files.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Train Dataset**

In [ ]:
# Define the path to the training dataset stored in Google Drive


TrainData_path = '/content/drive/My Drive/AircheckWorkshopData/Old/Trainset_Aircheck_WDR91_Hitgen_DEL.parquet'

# Read the Parquet file into a Pandas DataFrame
df_train = pd.read_parquet(TrainData_path, engine='pyarrow')

# Define the ratio of negative samples (N times more than positives)
N = 2  # Adjust this value as needed

# Select all rows where DELLabel == 1 (positive samples)
positive_samples = df_train[df_train["DELLabel"] == 1]

# Select N times more rows where DELLabel == 0 (negative samples)
negative_samples = df_train[df_train["DELLabel"] == 0].sample(n=len(positive_samples) * N, random_state=42)

# Combine both subsets to create a balanced dataset with the desired ratio
df_balanced = pd.concat([positive_samples, negative_samples])

# Define the output path with a descriptive filename
output_folder = os.path.dirname(TrainData_path)  # Same folder as the input file
output_filename = f"Trainset_Aircheck_WDR91_Hitgen_DEL_balanced_{N}xNeg.parquet"
output_path = os.path.join(output_folder, output_filename)

# Save the balanced dataset as a new Parquet file
df_balanced.to_parquet(output_path, index=False)

print(f"Balanced dataset saved as '{output_filename}' in '{output_folder}'")

df_train=df_balanced

In [ ]:
# Display first few rows of the train dataset
df_train.head()

,ID,DEL_ID,DELLabel,RawCount,Target,ECFP4,ECFP6,FCFP4,FCFP6,MACCS,RDK,AVALON,ATOMPAIR,TOPTOR,MW,ALOGP
0,22010205280358,L22-102-528-358,1,7,WDR91,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","12,1,3,0,17,0,3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","12,1,3,1,17,0,3,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,...","1,0,1,0,1,1,0,1,1,0,1,1,0,1,1,1,1,1,1,0,0,1,1,...","2,0,0,0,0,0,2,0,0,0,0,0,0,0,0,20,0,2,0,0,0,0,0...","0,1,0,0,4,0,0,1,0,0,0,4,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",499,3.4
1,49018705020137,L49-187-502-137,1,6,WDR91,"0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...","14,1,3,0,8,0,4,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0...","14,1,4,0,8,0,4,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0...","0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,...","1,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0...","0,0,0,0,2,0,0,0,0,0,0,5,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",458,1.0
2,49018703380329,L49-187-338-329,1,18,WDR91,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","9,1,2,0,11,0,6,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0...","9,1,3,0,11,0,6,0,1,0,0,0,0,0,0,0,0,0,0,2,0,0,0...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,1,0,0,1,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0,...","1,0,0,0,1,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",438,1.5
3,22008904630213,L22-89-463-213,1,7,WDR91,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...","13,2,2,0,15,0,3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...","13,2,2,0,15,0,3,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,...","1,1,0,1,1,1,0,1,1,1,1,1,1,0,1,0,1,0,0,1,0,1,1,...","0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,17,0,0,0,0,0,0,0...","1,0,0,0,3,0,0,0,0,0,0,2,2,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",466,3.4
4,23036500201654,L23-365-20-1654,1,12,WDR91,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","15,1,3,0,14,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...","15,1,3,0,14,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,0,0,0,1,0,1,1,1,1,1,0,0,0,1,1,0,0,1,0,1,1,...","0,0,0,0,6,0,1,0,0,0,0,0,0,0,0,7,0,2,0,0,0,0,0,...","0,0,0,0,2,0,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...",470,3.4


In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_train.columns.tolist()
print(column_names_list)

['ID', 'DEL_ID', 'DELLabel', 'RawCount', 'Target', 'ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR', 'MW', 'ALOGP']


### **Test Dataset**

In [ ]:
# Define the path to the test dataset stored in Google Drive
# Ensure that the file path is correct before running
TestData_path = '/content/drive/My Drive/AircheckWorkshopData/AircheckWorkshop2025_TestFile.csv'

# Load the CSV file into a Pandas DataFrame
df_test = pd.read_csv(TestData_path)

In [ ]:
# Display first few rows of the test dataset
df_test.head()

,Unnamed: 0,"﻿""Row Number""",ChemiReg ID,Supplier,KD (M),%Binding (Affinity) (%),experimental reliability,ID,SMILES (Compounds),ECFP4,ECFP6,FCFP4,FCFP6,MACCS,RDK,AVALON,TOPTOR,ATOMPAIR
0,0,1,DR003448a,HITGEN / JAMES,0.000007,103.91,high,C26H23N3O3,COc1c(C#N)cccc1CC(=O)Nc1ccc2c(c1)CCCN2C(=O)c1c...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","9,1,4,0,18,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0...","9,1,4,0,18,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,0,0,0,1,1,1,1,0,1,0,0,1,1,1,1,0,1,0,0,1,1,...","0,0,0,0,2,0,0,0,0,0,0,0,0,0,0,5,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,2,0,0,0,0,0,0,3,1,0,1,0,0,0,0,0,0,0,0,..."
1,1,12,DR003456a,HITGEN / JAMES,0.000005,116.69,high,C25H20N4OS,N#Cc1cccc2c1CC[C@H]2NC(=O)c1ccc(SCc2cn3ccccc3n...,"0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","6,1,3,0,19,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0...","6,1,3,0,19,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,1,0,0,1,0,0,0,0,1,1,0,1,1,1,1,0,1,0,0,1,1,...","0,0,0,0,0,0,2,0,0,0,0,0,0,0,1,8,0,4,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,1,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,..."
2,2,19,DR003457a,HITGEN / JAMES,0.000008,73.16,high,C24H21N3O2,COc1ccc(-c2cncc(C(=O)N[C@@H]3CCc4c(C#N)cccc43)...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","7,1,3,0,17,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0...","7,1,3,0,17,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,0,0,0,1,0,0,1,0,1,1,0,1,1,1,1,0,1,1,1,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,...","0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,2,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,..."
3,3,4,DR003460a,HITGEN / JAMES,0.000004,120.62,high,C25H19N5O,N#Cc1cccc2c1CC[C@H]2NC(=O)c1ccc(Nc2ncnc3ccccc2...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","5,1,2,0,20,0,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0...","5,1,2,0,20,0,3,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,0,0,0,1,0,1,0,0,1,1,1,0,1,1,1,0,1,0,0,0,1,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,1,0,0,0,0,0,0,3,0,0,0,0,0,0,0,0,0,0,0,..."
4,4,15,DR003461a,HITGEN / JAMES,0.000018,92.94,high,C24H23N5O3,CC(=O)c1cccc(OCc2cn(-c3ccc(C(=O)N4CCC(C#N)CC4)...,"0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","11,0,4,0,14,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","11,0,4,0,14,0,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","1,1,1,0,0,1,0,0,0,0,1,1,0,0,0,1,1,1,0,1,1,0,1,...","0,0,0,1,4,0,0,0,0,0,0,0,0,0,0,6,0,0,0,0,0,0,0,...","0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...","0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,1,..."


In [ ]:
# Get the list of column names from the DataFrame and print them
column_names_list = df_test.columns.tolist()
print(column_names_list)



---



# **🟣 3: Selecting Desired Columns (Finger prints) and ML labels**



In [ ]:
# Function to process fingerprint columns
def process_data0(X, column_name):
    """Split comma-separated fingerprint data into separate numeric columns."""
    X_split = X[column_name].str.split(',', expand=True)
    X_split = X_split.apply(pd.to_numeric, errors='coerce')
    X_split.columns = [f'{column_name}_{i}' for i in range(X_split.shape[1])]
    #return np.stack(X[column_name].apply(lambda x: np.fromstring(x, sep=',', dtype=np.float32)))
    return X_split

def process_data(X, column_name):
    return np.stack(X[column_name].apply(lambda x: np.fromstring(x, sep=',', dtype=np.float32)))

# List of available fingerprint column names
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
selected_fps = 'ECFP4'  # Replace with desired fingerprints

# Create TrainData and TestData using the selected fingerprints
TrainData = process_data(df_train, selected_fps)
TestData = process_data(df_test, selected_fps)

# Create TrainLabel from the 'DELLabel' column
TrainLabel = df_train['DELLabel']

# print(TrainData.head(), TestData.head())



---



# **🟣 4: Define the ML Model**

**LightGBM**

Light Gradient Boosting Machine is a highly efficient, scalable machine learning framework for gradient boosting, designed for speed and performance. It builds decision trees by optimizing the model through iterative training, focusing on minimizing errors in predictions. Unlike traditional gradient boosting algorithms, LightGBM uses a histogram-based approach that speeds up the training process, especially for large datasets. It is known for handling categorical features directly, offering better accuracy with less computational cost. LightGBM is widely used for classification, regression, and ranking tasks, making it popular in data science competitions and real-world applications.

In [ ]:
from lightgbm import LGBMClassifier

# Initialize model with detailed hyperparameters using default values
model = LGBMClassifier(
    n_estimators=100,  # Number of boosting iterations (trees)
    n_jobs=1,  # Number of parallel jobs (1 for no parallelism)
    learning_rate=0.1,  # Learning rate
    max_depth=-1,  # No limit on maximum depth of trees
    min_samples_leaf=20,  # Minimum samples at leaf node
    min_samples_split=2,  # Minimum samples to split node
    lambda_l2=0.0,  # L2 regularization (no regularization)
    lambda_l1=0.0,  # L1 regularization (no regularization)
    num_leaves=31,  # Number of leaves in each tree
    max_bin=255,  # Maximum number of bins
    subsample=1.0,  # Subsample ratio for training data (use all data)
    colsample_bytree=1.0,  # Subsample ratio for features (use all features)
    use_best_model=True,  # Use the best model based on validation performance
    random_state=None,  # Random seed for reproducibility (None for random)
    boosting_type='gbdt',  # Boosting type (Gradient Boosting Decision Tree)
    early_stopping_rounds=None,  # No early stopping
    min_split_gain=0.0,  # Minimum loss reduction required to make a further partition
    ignore_column_check=False  # Do not handle missing values automatically
)

# Model is now initialized with default hyperparameters




---



# **🟣 5: Training the Model and Evaluating Performance on the Cross-Validation Set**

###**Cross Validation**

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, cohen_kappa_score
)

# Function to train the model and compute all classification metrics
def train_model(CrossVal_data_train, CrossVal_data_test, CrossVal_label_train, CrossVal_label_test):
    """Train a LightGBM model and compute accuracy, precision, recall, F1, AUC, MCC, and Kappa."""
    model = LGBMClassifier(random_state=42)
    model.fit(CrossVal_data_train, CrossVal_label_train)

    y_pred = model.predict(CrossVal_data_test)
    y_scores = model.predict_proba(CrossVal_data_test)[:, 1]  # Probability for positive class

    metrics = {
        "Accuracy": accuracy_score(CrossVal_label_test, y_pred),
        "Precision": precision_score(CrossVal_label_test, y_pred, zero_division=0),
        "Recall": recall_score(CrossVal_label_test, y_pred),
        "F1-Score": f1_score(CrossVal_label_test, y_pred),
        "AUC-ROC": roc_auc_score(CrossVal_label_test, y_scores) if len(set(CrossVal_label_test)) > 1 else None,
        "MCC": matthews_corrcoef(CrossVal_label_test, y_pred),
        "Cohen's Kappa": cohen_kappa_score(CrossVal_label_test, y_pred),
    }

    return model, metrics

# Cross-validation
Nfold = 2
skf = StratifiedKFold(n_splits=Nfold, shuffle=True, random_state=42)
fold_metrics = []

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(TrainData, TrainLabel)):
    # Renaming the partitions inside the loop
    CrossVal_data_train, CrossVal_data_test = TrainData.iloc[train_idx], TrainData.iloc[test_idx]
    CrossVal_label_train, CrossVal_label_test = TrainLabel.iloc[train_idx], TrainLabel.iloc[test_idx]

    # Now using the renamed variables for both train and test data
    _, metrics = train_model(CrossVal_data_train, CrossVal_data_test, CrossVal_label_train, CrossVal_label_test)
    fold_metrics.append(metrics)

    # Print fold metrics
    print(f"Fold {fold_idx+1} Metrics:")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")
    print("-" * 100)

[LightGBM] [Info] Number of positive: 14389, number of negative: 14389
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.494223 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5985
[LightGBM] [Info] Number of data points in the train set: 28778, number of used features: 2032
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Fold 1 Metrics:
Accuracy: 0.8814
Precision: 0.8739
Recall: 0.8916
F1-Score: 0.8826
AUC-ROC: 0.9511
MCC: 0.7630
Cohen's Kappa: 0.7629
----------------------------------------
[LightGBM] [Info] Number of positive: 14389, number of negative: 14389
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.516967 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total 

In [ ]:
# Compute average metrics across folds
avg_metrics = {metric: np.mean([fold[metric] for fold in fold_metrics]) for metric in fold_metrics[0]}

# Print average metrics
print("\nAverage Metrics across all folds:")
for metric, value in avg_metrics.items():
    print(f"{metric}: {value:.4f}")


Average Metrics across all folds:
Accuracy: 0.8801
Precision: 0.8714
Recall: 0.8918
F1-Score: 0.8815
AUC-ROC: 0.9513
MCC: 0.7604
Cohen's Kappa: 0.7602


---

### **Train Final Model**



In [ ]:
import joblib
# Function to train the final model on the entire dataset
def train_final_model(X, y):
    final_model = LGBMClassifier(random_state=42)
    final_model.fit(X, y)
    model_filename = f"/content/drive/My Drive/AircheckWorkshopData/final_model.pkl"
    joblib.dump(final_model, model_filename)
    print(f"Final model saved as {model_filename}.")
    return final_model

# Train final model on entire dataset
final_model = train_final_model(TrainData, TrainLabel)

[LightGBM] [Info] Number of positive: 28778, number of negative: 28778
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.217855 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6286
[LightGBM] [Info] Number of data points in the train set: 57556, number of used features: 2045
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Final model saved as /content/drive/My Drive/AircheckWorkshopData/final_model.pkl.


---

# 🚀**Virtual Screening**🧬
![Alt Text](https://drive.google.com/uc?id=1OZDhx1wOLYu4Xy5CNZFsY7xntVe3mgmh)



# **🟣 6: Screen Test Compounds**
We can now screen new compounds to predict their activity. This involves passing SMILES strings through the trained model.

In [ ]:
def evaluate_model(model, X_test):
    """Evaluate the model on the test set and return predictions and probabilities."""
    y_scores = model.predict_proba(X_test)[:, 1]  # Probability for positive class
    return np.round(y_scores, 3)

predictions = evaluate_model(final_model, TestData)

# Create a DataFrame with SMILES and prediction scores
prediction_df = pd.DataFrame({
    'SMILES': df_test["SMILES"],  # Assuming 'SMILES' is a column in df_test
    'Prediction_Score': predictions
})

# Sort the DataFrame by prediction score in descending order
prediction_df_sorted = prediction_df.sort_values(by='Prediction_Score', ascending=False)

# Keep only those with score > 0.7 as Possible Nominees
nominees = prediction_df_sorted[prediction_df_sorted['Prediction_Score'] > 0.5]

# Get the number of nominees
num_nominees = nominees.shape[0]
print(f"\nNumber of Possible Nominees: {num_nominees}")

# Print the top 10 highest-ranked predictions
print("Top 10 Predictions:")
print(prediction_df_sorted.head(10))


Number of Possible Nominees: 91
Top 10 Predictions:
                                                 SMILES  Prediction_Score
514   N#Cc1cccc(OCc2cn(-c3ccc(C(=O)N4CCC(C#N)CC4)cc3...             0.967
9190  CC(=O)c1cccc(OCc2cn(-c3ccc(C(=O)N4CCC(C#N)CC4)...             0.964
614   N#Cc1cccc2c1CC[C@H]2NC(=O)c1ccc(SCc2cn3ccccc3n...             0.923
6818  N#Cc1cccc2c1CC[C@H]2NC(=O)c1ccc(S(=O)(=O)NCc2c...             0.922
9648  N#Cc1cccc2c1CC[C@H]2NC(=O)c1ccc(Nc2ncnc3ccccc2...             0.918
8540  CCN(CC)Cc1cn(Cc2ccc(C(=O)N[C@@H]3CCc4c(C#N)ccc...             0.821
5429     N#Cc1cccc2c1CC[C@H]2NC(=O)c1ccc(-c2ccco2)[nH]1             0.807
569    Cc1ccc(NC(=O)C(Sc2nnc(-c3ccco3)o2)c2ccccc2)cc1Cl             0.777
4556                   N#Cc1ccc(C(=O)N2CCc3ccccc3C2)cc1             0.775
698   COc1ccc(-c2cncc(C(=O)N[C@@H]3CCc4c(C#N)cccc43)...             0.755


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


---

# **🟣 6: Using Ensemble of Models**

This method improves prediction reliability by training multiple models, each based on a different molecular fingerprint (e.g., ECFP4, FCFP6, MACCS). Instead of relying on a single model, we evaluate test data across all trained models, compute the mean prediction score, standard deviation, and confidence score. The final score, calculated as mean prediction minus standard deviation, helps select high-confidence nominees while reducing uncertainty. This approach ensures more robust and reliable predictions compared to using a single fingerprint-based model.

In [ ]:
import numpy as np
import pandas as pd
import joblib
from lightgbm import LGBMClassifier

# Function to process data
def process_data(X, column_name):
    return np.stack(X[column_name].apply(lambda x: np.fromstring(x, sep=',', dtype=np.float32)))

# List of fingerprint columns to train on
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']
fingerprint_columns = ['ECFP4', 'MACCS', 'RDK', ]

# Dictionary to store trained models
trained_models = {}

# Train models for each fingerprint column
for fp in fingerprint_columns:
    print(f"Training model for {fp}...")
    TrainData = process_data(df_train, fp)
    TrainLabel = df_train['DELLabel']

    model = LGBMClassifier(random_state=42)
    model.fit(TrainData, TrainLabel)

    model_filename = f"/content/drive/My Drive/AircheckWorkshopData/final_model_{fp}.pkl"
    joblib.dump(model, model_filename)
    print(f"Model for {fp} saved as {model_filename}.")

    trained_models[fp] = model

# Function to evaluate models
def evaluate_model(model, X_test):
    return model.predict_proba(X_test)[:, 1]  # Probability for positive class

# Store predictions for each model
all_predictions = {}

for fp in fingerprint_columns:
    print(f"Evaluating model for {fp}...")
    TestData = process_data(df_test, fp)
    all_predictions[fp] = evaluate_model(trained_models[fp], TestData)

# Convert predictions to DataFrame
predictions_df = pd.DataFrame(all_predictions)
predictions_df['Mean_Prediction'] = predictions_df.mean(axis=1)
predictions_df['Std_Dev'] = predictions_df.std(axis=1)
predictions_df['Confidence_Score'] = 1 - predictions_df['Std_Dev']  # Higher means more confident
predictions_df['Final_Score'] = predictions_df['Mean_Prediction'] - predictions_df['Std_Dev']

# Add SMILES column
predictions_df['SMILES'] = df_test['SMILES']

# Sort by Final Score
predictions_df_sorted = predictions_df.sort_values(by='Final_Score', ascending=False)

# Select nominees with Final Score > 0.5
nominees = predictions_df_sorted[predictions_df_sorted['Final_Score'] > 0.5]
num_nominees = nominees.shape[0]

print(f"\nNumber of Possible Nominees: {num_nominees}")
print("Top 10 Predictions:")
print(predictions_df_sorted.head(10))

Training model for ECFP4...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 28778, number of negative: 28778
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.230845 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 6286
[LightGBM] [Info] Number of data points in the train set: 57556, number of used features: 2045
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Model for ECFP4 saved as /content/drive/My Drive/AircheckWorkshopData/final_model_ECFP4.pkl.
Training model for MACCS...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Info] Number of positive: 28778, number of negative: 28778
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078067 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 266
[LightGBM] [Info] Number of data points in the train set: 57556, number of used features: 133
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Model for MACCS saved as /content/drive/My Drive/AircheckWorkshopData/final_model_MACCS.pkl.
Training model for RDK...


...

Evaluating model for MACCS...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Evaluating model for RDK...


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(



Number of Possible Nominees: 7
Top 10 Predictions:
         ECFP4     MACCS       RDK  Mean_Prediction   Std_Dev  \
9648  0.917826  0.719573  0.927076         0.854825  0.095712   
6818  0.921937  0.671465  0.909300         0.834234  0.115211   
698   0.754894  0.682686  0.910106         0.782562  0.094883   
7345  0.650973  0.722993  0.845103         0.739690  0.080128   
5429  0.807495  0.586360  0.950005         0.781287  0.149610   
614   0.923092  0.431160  0.860902         0.738385  0.218719   
9190  0.964090  0.407727  0.892912         0.754910  0.247209   
7031  0.652313  0.516231  0.505827         0.558124  0.066737   
5031  0.461614  0.552118  0.543571         0.519101  0.040799   
8971  0.475232  0.473584  0.513462         0.487426  0.018423   

      Confidence_Score  Final_Score  \
9648          0.904288     0.759113   
6818          0.884789     0.719023   
698           0.905117     0.687679   
7345          0.919872     0.659561   
5429          0.850390     0.631677  



---



In [ ]:
# Select nominees with Final Score > 0.5
nominees = predictions_df_sorted[predictions_df_sorted['Final_Score'] > 0.3]
num_nominees = nominees.shape[0]

print(f"\nNumber of Possible Nominees: {num_nominees}")
print("Top 10 Predictions:")
print(predictions_df_sorted.head(10))


Number of Possible Nominees: 71
Top 10 Predictions:
         ECFP4     MACCS       RDK  Mean_Prediction   Std_Dev  \
9648  0.917826  0.719573  0.927076         0.854825  0.095712   
6818  0.921937  0.671465  0.909300         0.834234  0.115211   
698   0.754894  0.682686  0.910106         0.782562  0.094883   
7345  0.650973  0.722993  0.845103         0.739690  0.080128   
5429  0.807495  0.586360  0.950005         0.781287  0.149610   
614   0.923092  0.431160  0.860902         0.738385  0.218719   
9190  0.964090  0.407727  0.892912         0.754910  0.247209   
7031  0.652313  0.516231  0.505827         0.558124  0.066737   
5031  0.461614  0.552118  0.543571         0.519101  0.040799   
8971  0.475232  0.473584  0.513462         0.487426  0.018423   

      Confidence_Score  Final_Score  \
9648          0.904288     0.759113   
6818          0.884789     0.719023   
698           0.905117     0.687679   
7345          0.919872     0.659561   
5429          0.850390     0.631677 

# **🟣 7: Apply Medicinal Chemistry Filters**

This partapplies drug-likeness filters to the selected nominees based on three key rules: Lipinski, Ghose, and Veber. The molecules that pass all three filters are retained as final candidates.

- **Lipinski’s Rule of 5**: Evaluates drug-likeness based on molecular weight, lipophilicity (logP), hydrogen bond donors/acceptors, and rotatable bonds.
- **Ghose Filter**: Checks molecular weight, logP, number of atoms, and molar refractivity to ensure favorable pharmacokinetics.
- **Veber Rule**: Ensures compounds have limited rotatable bonds and acceptable topological polar surface area for good oral bioavailability.

**Note:** This is a simplified version of many available drug-like property filters!

In [ ]:
from rdkit import Chem
import rdkit.Chem.Descriptors as Descriptors

class SimplifiedDrugFilters:
    def __init__(self):
        pass

    @staticmethod
    def fetch_attributes(molecule):
        return {
            "molecular_weight": Descriptors.ExactMolWt(molecule),
            "logp": Descriptors.MolLogP(molecule),
            "h_bond_donor": Descriptors.NumHDonors(molecule),
            "h_bond_acceptors": Descriptors.NumHAcceptors(molecule),
            "rotatable_bonds": Descriptors.NumRotatableBonds(molecule),
            "num_atoms": Chem.rdchem.Mol.GetNumAtoms(molecule),
            "molar_refractivity": Chem.Crippen.MolMR(molecule),
            "topo_surface_area": Chem.QED.properties(molecule).PSA
        }

    def filter(self, smiles):
        results = {"lipinski": [], "ghose": [], "veber": [], "pass_all_filters": []}
        molecules = [Chem.MolFromSmiles(i) for i in smiles]

        for i, mol in enumerate(molecules):
            props = self.fetch_attributes(mol)

            # Lipinski Rule of 5
            lipinski = (props["molecular_weight"] <= 500 and props["logp"] <= 5 and
                        props["h_bond_donor"] <= 5 and props["h_bond_acceptors"] <= 10 and
                        props["rotatable_bonds"] <= 5)

            # Ghose Filter
            ghose = (160 <= props["molecular_weight"] <= 480 and -0.4 <= props["logp"] <= 5.6 and
                     20 <= props["num_atoms"] <= 70 and 40 <= props["molar_refractivity"] <= 130)

            # Veber Rule
            veber = (props["rotatable_bonds"] <= 10 and props["topo_surface_area"] <= 140)

            results["lipinski"].append(lipinski)
            results["ghose"].append(ghose)
            results["veber"].append(veber)
            results["pass_all_filters"].append(all([lipinski, ghose, veber]))

        return results

In [ ]:
# Apply drug design filters to the selected nominees
filter = SimplifiedDrugFilters()
filter_results = pd.DataFrame(filter.filter(nominees["SMILES"].tolist()), index=nominees.index)

# Merge the filter results with nominees
nominees_filtered = pd.merge(nominees, filter_results, left_index=True, right_index=True)
print("Top 10 Predictions:")
print(nominees_filtered.head(10))

nominees_filtered = nominees_filtered[nominees_filtered["pass_all_filters"] == True]
num_nominees_filtered = nominees_filtered.shape[0]
print(f"\nNumber of Filtered Nominees: {num_nominees_filtered}")

Top 10 Predictions:
         ECFP4     MACCS       RDK  Mean_Prediction   Std_Dev  \
9648  0.917826  0.719573  0.927076         0.854825  0.095712   
6818  0.921937  0.671465  0.909300         0.834234  0.115211   
698   0.754894  0.682686  0.910106         0.782562  0.094883   
7345  0.650973  0.722993  0.845103         0.739690  0.080128   
5429  0.807495  0.586360  0.950005         0.781287  0.149610   
614   0.923092  0.431160  0.860902         0.738385  0.218719   
9190  0.964090  0.407727  0.892912         0.754910  0.247209   
7031  0.652313  0.516231  0.505827         0.558124  0.066737   
5031  0.461614  0.552118  0.543571         0.519101  0.040799   
8971  0.475232  0.473584  0.513462         0.487426  0.018423   

      Confidence_Score  Final_Score  \
9648          0.904288     0.759113   
6818          0.884789     0.719023   
698           0.905117     0.687679   
7345          0.919872     0.659561   
5429          0.850390     0.631677   
614           0.781281     0.5

---

# **🟣 8: Cluster and Select Nominated Compounds for Further Laboratory Test**
Finally, we use similarity-based clustering to identify diverse candidates among the top hits.

### 1. **Generate Molecular Fingerprints**:
   - Convert each molecule (represented by its SMILES string) into a **Morgan fingerprint** using RDKit's `AllChem.GetMorganFingerprintAsBitVect`. This produces a binary vector that encodes the molecular structure.

### 2. **Cluster Molecules Using LeaderPicker**:
   - Apply the **LeaderPicker** algorithm to the fingerprints to identify "leader" molecules. These leaders act as centroids for clusters of similar molecules. The `thresh` parameter determines the minimum similarity between a leader and other molecules for them to belong to the same cluster.

### 3. **Assign Molecules to Clusters**:
   - After identifying the leaders, calculate the **Tanimoto similarity** between each molecule's fingerprint and the fingerprints of the leader molecules. Each molecule is then assigned to the cluster whose leader it is most similar to. The function `assignPointsToClusters` groups molecules based on their similarity to these leaders.

### 4. **Select Representative Molecules and Sort**:
   - Within each cluster, select a subset of molecules (typically 1/20th of the cluster size). The selection is based on the similarity to previously selected molecules, ensuring diversity within the cluster. Finally, the results are sorted by **Prediction Score** and **Cluster ID** to highlight the most relevant molecules.


In [ ]:
from rdkit.SimDivFilters import rdSimDivPickers
from rdkit import DataStructs
from rdkit.Chem import AllChem
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from rdkit import Chem

def assignPointsToClusters(picks, fps):
    clusters = defaultdict(list)
    for i, idx in enumerate(picks):
        clusters[i].append(idx)
    sims = np.zeros((len(picks), len(fps)))
    for i in tqdm(range(len(picks))):
        pick = picks[i]
        sims[i, :] = DataStructs.BulkTanimotoSimilarity(fps[pick], fps)
        sims[i, i] = 0  # Don't compare the molecule with itself
    best = np.argmax(sims, axis=0)
    for i, idx in enumerate(best):
        if i not in picks:
            clusters[idx].append(i)
    return clusters

# Assuming TestData is a DataFrame containing SMILES
# Assuming prediction_df contains the predictions and their corresponding "Prediction_Score"

# Generate Morgan fingerprints using AllChem
fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smi), radius=3, nBits=2048) for smi in tqdm(nominees_filtered["SMILES"])]

  0%|          | 0/57 [00:00<?, ?it/s][21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECATION WARNING: please use MorganGenerator
[21:25:43] DEPRECA

In [ ]:
# Perform clustering using LeaderPicker
lp = rdSimDivPickers.LeaderPicker()
thresh = 0.65  # Minimum distance between cluster centroids
picks = lp.LazyBitVectorPick(fps, len(fps), thresh)  # Using the ExplicitBitVect fingerprints from AllChem
clusters = assignPointsToClusters(picks, fps)

# Assign cluster ids to the prediction_df based on the indices from the clusters
cluster_ids = np.zeros(len(nominees_filtered))  # Initialize cluster_ids for the entire prediction_df

# Make sure to correctly assign the cluster ids
for key, val in clusters.items():
    cluster_ids[val] = key  # Assign the cluster ID to the correct indices

# Add the cluster ids to the prediction_df
nominees_filtered['cluster_id'] = cluster_ids

# Sort the results by Prediction Score and Cluster ID
#nominees_filtered.sort_values(by=["Final_Score", "cluster_id"], ascending=[False, True], inplace=True)

num_clusters = len(set(nominees_filtered["cluster_id"]))
print(f"Number of clusters generated: {num_clusters}")

# Sort the dataframe by Final_Score in descending order
nominees_filtered.sort_values(by=["Final_Score"], ascending=False, inplace=True)

# Select one nominee per cluster: the one with the highest score
best_nominees = nominees_filtered.groupby("cluster_id").first().reset_index()

# Print the selected nominees (one per cluster)
print(best_nominees.head(10))

NameError: name 'rdSimDivPickers' is not defined

I will edit this:

## **How to submit results**
save prediction in CSV format in our GCP bucket
With there TeamName like Team1.csv

Other ideas that I may add:

- Confusion matrix and other visualizations for this part
- Introducing other ML models they can use
- MLflow?
- Dummy test to show importance of mertics
- Add metrics to the ensemble one.

!!!! For the complains in the Bootcamp we had (simpler models, better results) --> We may want to give people some development data